[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bulentsoykan/ABM-particle-filter-notebooks/blob/main/notebook_04_degeneracy.ipynb)

# Module 4: Particle Degeneracy

**Learning objectives**
1. Understand what particle degeneracy is and when it occurs
2. Visualize the weight collapse that leads to degeneracy
3. See how resampling (and jitter) mitigates the problem
4. Use Effective Sample Size (ESS) as a diagnostic


---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,  'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'axes.grid': True,
    'font.size': 11,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded.")

In [ ]:
def systematic_resample(weights):
    '''Systematic resampling — O(N), low variance, preferred for PF.

    Algorithm (Eq. 5-6 in Malleson et al. 2020):
      1. Draw one random U ~ Uniform[0, 1/N]
      2. Space N points evenly: U, U+1/N, U+2/N, ..., U+(N-1)/N
      3. Walk the CDF; each particle covers a slice proportional to its weight
         -> heavy particles get sampled multiple times, light ones get dropped
    '''
    N   = len(weights)
    w   = np.asarray(weights, dtype=float)
    w   = w / w.sum()
    U   = np.random.uniform(0, 1.0 / N)
    pts = (np.arange(N) + U) / N
    cdf = np.cumsum(w)
    i, j, idxs = 0, 0, np.zeros(N, dtype=int)
    while i < N:
        if pts[i] <= cdf[j]:
            idxs[i] = j
            i += 1
        else:
            j = min(j + 1, N - 1)
    return idxs

def ess(weights):
    '''Effective Sample Size — diagnostic for particle degeneracy.
    ESS = 1 / sum(w_i^2).  Ranges from 1 (total degeneracy) to N (uniform weights).
    Rule of thumb: resample when ESS < N/2.
    '''
    w = np.asarray(weights, float)
    w = w / w.sum()
    return 1.0 / np.sum(w**2)

print(f"systematic_resample and ess defined.")


## Part 1: Weight Collapse Without Resampling

Run the particle filter **without** the resampling step.
Watch the weights: after a few observations, almost all the probability
mass concentrates on one particle.  This is **particle degeneracy**.


In [ ]:
np.random.seed(0)

# Simple 1D tracking: object moves right at speed 1
N_P_DEG = 50
T_DEG   = 20

obj_x = np.arange(T_DEG, dtype=float)   # x = 0, 1, 2, ..., 19

# Initialise particles
p_nores  = np.random.uniform(-5, 10, N_P_DEG)   # no-resample run
p_resamp = np.random.uniform(-5, 10, N_P_DEG)   # resample run
w_nores  = np.ones(N_P_DEG) / N_P_DEG
w_resamp = np.ones(N_P_DEG) / N_P_DEG

history_w_nores  = [w_nores.copy()]
history_w_resamp = [w_resamp.copy()]
history_ess_no   = [N_P_DEG]
history_ess_re   = [N_P_DEG]

for t in range(1, T_DEG):
    obs = obj_x[t] + np.random.randn() * 0.3

    # ── No resample ──────────────────────────────────────────────────────────
    p_nores   += np.random.randn(N_P_DEG) * 0.8
    dists_no   = np.abs(p_nores - obs) + 1e-6
    lhood_no   = np.exp(-0.5 * (dists_no / 0.3)**2)
    w_nores   *= lhood_no
    w_nores   /= w_nores.sum()

    # ── With resample ─────────────────────────────────────────────────────────
    p_resamp  += np.random.randn(N_P_DEG) * 0.8
    dists_re   = np.abs(p_resamp - obs) + 1e-6
    lhood_re   = np.exp(-0.5 * (dists_re / 0.3)**2)
    w_resamp  *= lhood_re
    w_resamp  /= w_resamp.sum()
    # Resample!
    idxs       = systematic_resample(w_resamp)
    p_resamp   = p_resamp[idxs]
    w_resamp   = np.ones(N_P_DEG) / N_P_DEG

    history_w_nores.append(w_nores.copy())
    history_w_resamp.append(w_resamp.copy())
    history_ess_no.append(ess(w_nores))
    history_ess_re.append(ess(w_resamp))

print("Simulation complete.")
print(f"\nNo-resample  ESS at step 0: {history_ess_no[0]:.0f}")
print(f"No-resample  ESS at step {T_DEG-1}: {history_ess_no[-1]:.2f}  <- near 1 = DEGENERATE")
print(f"\nWith-resample ESS at step 0: {history_ess_re[0]:.0f}")
print(f"With-resample ESS at step {T_DEG-1}: {history_ess_re[-1]:.0f}  <- stays high = HEALTHY")


In [ ]:
# ── Visualise weight evolution ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Weight Evolution: No Resampling (top) vs With Resampling (bottom)',
             fontsize=14, fontweight='bold')

show_steps = [0, 5, 10, 19]

for col, t in enumerate(show_steps):
    # Top row: no resample
    ax_t = axes[0, col]
    ax_t.bar(np.arange(N_P_DEG), sorted(history_w_nores[t], reverse=True),
             color='#f85149', alpha=0.8, width=1.0)
    ax_t.set(title=f't = {t}', xlabel='Particle (sorted)',
             ylabel='Weight' if col == 0 else '')
    max_w = max(history_w_nores[t])
    ax_t.text(0.98, 0.97, f'max w = {max_w:.3f}\nESS = {history_ess_no[t]:.1f}',
              transform=ax_t.transAxes, ha='right', va='top', fontsize=9,
              bbox=dict(boxstyle='round', fc='#f8514922', ec='#f85149'))
    if col == 0:
        ax_t.set_ylabel('Weight (NO resample)', color='#f85149')

    # Bottom row: with resample
    ax_b = axes[1, col]
    ax_b.bar(np.arange(N_P_DEG), sorted(history_w_resamp[t], reverse=True),
             color='#3fb950', alpha=0.8, width=1.0)
    ax_b.set(title=f't = {t}', xlabel='Particle (sorted)',
             ylabel='Weight' if col == 0 else '')
    if col == 0:
        ax_b.set_ylabel('Weight (WITH resample)', color='#3fb950')

plt.tight_layout()
plt.savefig('figures/fig_04a_weight_collapse.png', dpi=150, bbox_inches='tight')
plt.show()
print("Without resampling: all weight concentrates on ONE particle (degeneracy!)")
print("With resampling:    weights stay uniform after each resample step.")

## Part 2: Effective Sample Size (ESS)

The ESS tells us how many *effectively independent* particles we have:

$$ESS = \frac{1}{\sum_i (w_i)^2}$$

- ESS = N (all weights equal): **healthy diversity**
- ESS = 1 (one particle has all weight): **full degeneracy**

As a rule of thumb, resample when ESS < N/2.


In [ ]:
fig, (ax_ess, ax_maxw) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Effective Sample Size (ESS) — Diagnostic for Degeneracy', fontsize=14, fontweight='bold')

steps = np.arange(T_DEG)

ax_ess.plot(steps, history_ess_no, color='#f85149', lw=2.5, marker='o', ms=5,
            label='No resampling')
ax_ess.plot(steps, history_ess_re, color='#3fb950', lw=2.5, marker='o', ms=5,
            label='With resampling')
ax_ess.axhline(N_P_DEG,     color='#8b949e', ls='--', lw=1, label=f'N = {N_P_DEG} (ideal)')
ax_ess.axhline(N_P_DEG / 2, color='#f0883e', ls=':',  lw=1.5, label=f'N/2 = {N_P_DEG//2} (resample threshold)')
ax_ess.set(xlabel='Time step', ylabel='ESS', title='ESS over time', ylim=(0, N_P_DEG * 1.1))
ax_ess.legend(fontsize=9)

max_w_no = [w.max() for w in history_w_nores]
max_w_re = [w.max() for w in history_w_resamp]
ax_maxw.plot(steps, max_w_no, color='#f85149', lw=2.5, marker='o', ms=5,
             label='No resampling')
ax_maxw.plot(steps, max_w_re, color='#3fb950', lw=2.5, marker='o', ms=5,
             label='With resampling')
ax_maxw.axhline(1.0 / N_P_DEG, color='#8b949e', ls='--', lw=1, label='1/N (ideal)')
ax_maxw.set(xlabel='Time step', ylabel='Max particle weight',
            title='Maximum Weight over Time')
ax_maxw.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig_04b_ess.png', dpi=150, bbox_inches='tight')
plt.show()
print("Without resampling: max weight -> 1.0 (one particle dominates), ESS -> 1")
print("With resampling:    max weight stays near 1/N,                   ESS ~ N")

## Part 3: Particle Noise (Jitter / Roughening)

After resampling, many particles are *identical* copies of the best particle.
Adding small Gaussian noise ("jitter") restores diversity.

But too much noise pushes particles far from the true state.
The optimal noise level is a key hyperparameter (paper uses sigma_p = 0.25).

First we define the ABM and PF needed for this comparison:


In [ ]:
# ── Minimal StationSim-style corridor ABM ──────────────────────────────────────

class Agent:
    '''A pedestrian moving left to right through a corridor.

    The ONLY source of randomness: when a faster agent catches up to a slower one,
    it randomly dodges LEFT or RIGHT (50/50 coin flip).  Everything else is
    deterministic.  That one binary choice is all it takes to cause divergence.
    '''
    def __init__(self, agent_id, env_width=100, env_height=50, rng=None):
        rng = rng or np.random
        self.id         = agent_id
        self.x          = rng.uniform(0, 3)
        self.y          = rng.uniform(5, env_height - 5)
        self.max_speed  = max(0.3, rng.normal(1.0, 0.4))
        self.env_width  = env_width
        self.env_height = env_height
        self.active     = True
        self.traj_x     = [self.x]
        self.traj_y     = [self.y]

    def step(self, all_agents, separation=4.5, rng=None):
        rng = rng or np.random
        if not self.active:
            self.traj_x.append(self.x)
            self.traj_y.append(self.y)
            return
        speed, dy = self.max_speed, 0.0
        for other in all_agents:
            if other.id == self.id or not other.active:
                continue
            dx   = other.x - self.x
            dist = np.hypot(dx, other.y - self.y)
            if 0 < dx < separation * 2 and dist < separation:
                speed *= 0.6
                dy = 1.0 if rng.random() < 0.5 else -1.0   # THE random choice
                break
        self.x = min(self.x + speed, self.env_width)
        self.y = np.clip(self.y + dy, 2, self.env_height - 2)
        if self.x >= self.env_width:
            self.active = False
        self.traj_x.append(self.x)
        self.traj_y.append(self.y)


class CorridorABM:
    '''Corridor where N agents walk left-to-right.

   State vector: [x0,y0, x1,y1, ..., x_{N-1},y_{N-1}]  — dimension 2*N_agents.
    '''
    def __init__(self, n_agents=10, env_width=100, env_height=50, seed=None):
        rng             = np.random.RandomState(seed)
        self.n_agents   = n_agents
        self.env_width  = env_width
        self.env_height = env_height
        self.rng        = rng
        self.agents     = [Agent(i, env_width, env_height, rng)
                           for i in range(n_agents)]
        self.pos_history = [self.get_positions()]
        self.step_count  = 0
        self.collisions  = 0

    def get_positions(self):
        return np.array([[a.x, a.y] for a in self.agents])

    def get_state_vector(self):
        '''Flat [x0,y0, x1,y1,...] — used by the particle filter.'''
        return self.get_positions().flatten()

    def step(self):
        before = [a.active for a in self.agents]
        for agent in self.agents:
            agent.step(self.agents, rng=self.rng)
        # count collision events (agents that slowed down)
        for a in self.agents:
            for b in self.agents:
                if a.id >= b.id or not a.active or not b.active:
                    continue
                if np.hypot(a.x - b.x, a.y - b.y) < 4.5:
                    self.collisions += 1
        self.pos_history.append(self.get_positions())
        self.step_count += 1

    def run(self, n_steps):
        for _ in range(n_steps):
            self.step()
        return self

print("CorridorABM defined.  State dimension = 2 x N_agents.")


In [ ]:
def systematic_resample(weights):
    '''Systematic resampling — O(N), low variance, preferred for PF.

    Algorithm :
      1. Draw one random U ~ Uniform[0, 1/N]
      2. Space N points evenly: U, U+1/N, U+2/N, ..., U+(N-1)/N
      3. Walk the CDF; each particle covers a slice proportional to its weight
         -> heavy particles get sampled multiple times, light ones get dropped
    '''
    N   = len(weights)
    w   = np.asarray(weights, dtype=float)
    w   = w / w.sum()
    U   = np.random.uniform(0, 1.0 / N)
    pts = (np.arange(N) + U) / N
    cdf = np.cumsum(w)
    i, j, idxs = 0, 0, np.zeros(N, dtype=int)
    while i < N:
        if pts[i] <= cdf[j]:
            idxs[i] = j
            i += 1
        else:
            j = min(j + 1, N - 1)
    return idxs

from copy import deepcopy

class ParticleFilterABM:
    '''SIR Particle Filter applied to CorridorABM.

    Each 'particle' is a full ABM instance.  Every resample_window steps:
      PREDICT  -> advance all N_p ABMs one step
      REWEIGHT -> score each particle against the pseudo-truth observation
      RESAMPLE -> duplicate high-weight particles, discard low-weight ones
      JITTER   -> add small Gaussian noise to prevent particle deprivation

    Reference: Malleson et al. (2020), Equations (3)-(7), Sections 3.5-3.13
    '''
    def __init__(self, n_particles, n_agents, env_width=100, env_height=50,
                 particle_std=0.25, resample_window=20):
        self.n_p    = n_particles
        self.n_a    = n_agents
        self.pstd   = particle_std
        self.window = resample_window
        self.particles = [
            CorridorABM(n_agents, env_width, env_height, seed=i)
            for i in range(n_particles)
        ]
        self.weights     = np.ones(n_particles) / n_particles
        self.w_history   = []    # weights at each update (for degeneracy viz)
        self.err_history = []    # mean weighted error at each update
        self.step_count  = 0

    def predict(self):
        for p in self.particles:
            p.step()
        self.step_count += 1

    def update(self, observation):
        '''Reweight + resample against observed state vector.'''
        # Eq.(3): epsilon_i = ||y_k - x_k^i||_2
        errors = np.array([
            np.linalg.norm(p.get_state_vector() - observation)
            for p in self.particles
        ])
        # Weight: w_i = eta / (1e-9 + epsilon_i)
        self.weights = 1.0 / (1e-9 + errors)
        self.weights /= self.weights.sum()
        self.w_history.append(self.weights.copy())
        self.err_history.append(float(np.dot(self.weights, errors)))

        # Systematic resample
        idxs = systematic_resample(self.weights)
        self.particles = [deepcopy(self.particles[k]) for k in idxs]
        self.weights   = np.ones(self.n_p) / self.n_p

        # Jitter / roughening: prevents all particles collapsing to one state
        if self.pstd > 0:
            for p in self.particles:
                noise = np.random.randn(self.n_a, 2) * self.pstd
                for k, ag in enumerate(p.agents):
                    ag.x = np.clip(ag.x + noise[k, 0], 0, p.env_width)
                    ag.y = np.clip(ag.y + noise[k, 1], 2, p.env_height - 2)

    def get_estimate(self):
        '''Weighted mean state estimate (Eq. 7).'''
        states = np.array([p.get_state_vector() for p in self.particles])
        return np.average(states, weights=self.weights, axis=0)

    def best_particle(self):
        return self.particles[np.argmax(self.weights)]

    def run(self, truth_model, n_steps):
        '''Run PF alongside the truth model for n_steps.'''
        for t in range(n_steps):
            self.predict()
            truth_model.step()
            if (t + 1) % self.window == 0:
                self.update(truth_model.get_state_vector())
        return self

print("ParticleFilterABM defined.")


In [ ]:
# Compare 4 noise levels on the same ABM+PF run
from copy import deepcopy

noise_levels = [0.0, 0.1, 0.25, 0.5]   # sigma_p in the paper
N_A_NOISE, N_P_NOISE, N_S_NOISE = 8, 100, 60

noise_errors = {}
for sigma in noise_levels:
    run_errs = []
    for seed in range(5):
        truth_n = CorridorABM(n_agents=N_A_NOISE, seed=seed)
        pf_n    = ParticleFilterABM(n_particles=N_P_NOISE, n_agents=N_A_NOISE,
                                     particle_std=sigma, resample_window=20)
        pf_n.run(truth_n, N_S_NOISE)
        run_errs.append(pf_n.err_history)
    noise_errors[sigma] = np.array(run_errs)
    med = np.median(noise_errors[sigma][:, -1])
    print(f"  sigma_p = {sigma:.2f}  |  median final error: {med:.2f}")

print("\nOptimal noise level (from paper): sigma_p = 0.25")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
fig.suptitle('Effect of Particle Noise (sigma_p) on PF Error', fontsize=14, fontweight='bold')
colors_n = ['#f85149', '#f0883e', '#3fb950', '#58a6ff']

for ax, (sigma, errs), col in zip(axes, noise_errors.items(), colors_n):
    windows = np.arange(1, errs.shape[1] + 1)
    med = np.median(errs, axis=0)
    lo  = np.percentile(errs, 25, axis=0)
    hi  = np.percentile(errs, 75, axis=0)
    ax.fill_between(windows, lo, hi, alpha=0.25, color=col)
    ax.plot(windows, med, color=col, lw=2.5, marker='o', ms=6)
    ax.set(xlabel='DA window', ylabel='Mean PF error' if sigma == 0.0 else '',
           title=f'sigma_p = {sigma}')
    verdict = ('NO diversity!' if sigma == 0.0 else
               'Slightly under' if sigma == 0.1 else
               'Optimal (paper)' if sigma == 0.25 else
               'Too noisy')
    ax.text(0.05, 0.95, verdict, transform=ax.transAxes, va='top', fontsize=9,
            color=col, fontweight='bold',
            bbox=dict(boxstyle='round', fc='#161b22', ec=col, alpha=0.9))

plt.tight_layout()
plt.savefig('figures/fig_04c_noise_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Weight collapse** | ESS -> 1; one particle dominates | Resample when ESS < N/2 |
| **Particle deprivation** | All particles identical after resample | Add jitter (noise sigma_p) |
| **Too much noise** | Particles drift far from true state | Tune sigma_p (paper uses 0.25) |

**The trade-off**: diversity vs accuracy
- Too little noise → deprivation (cloned particles)
- Too much noise → accuracy loss (particles wander)
- Just right (sigma_p ≈ 0.25 in this model) → maintains diversity without losing accuracy

**Next: Module 5** — the fundamental scaling challenge:
why exponentially more particles are needed as the number of agents grows.
